# FARSI_WHAM SuDo-RM-RF (GroupComm v2) — Colab training

Multi-session-friendly training notebook. Design:

- **Local disk every session (ephemeral Place):** HuggingFace downloads + audio caches rebuilt from scratch each session under `/data` — download and decode are cheap, we do not persist the ~24 GB audio cache on Google Drive.
- **Google Drive (persistent):** only checkpoints, experiment audio logs, and the per-epoch `metrics.jsonl`. These are small and are written/read from Drive directly during training.
- **Resume capability:** `latest_checkpoint.pt` is stored on the Drive and training resumes with `-rfc latest` automatically when it exists.

Runtime -> **GPU** (T4 is enough).

## 1. GPU check

In [ ]:
import os
os.system('nvidia-smi')

## 2. Install dependencies

Colab already ships torch; we only need a few lighter pieces.

In [ ]:
import os
os.system('pip -q install glob2 scipy soundfile tqdm datasets musdb')

## 3. Get the repo

Set `REPO_URL` to your repo clone address. Re-running this cell keeps the repo up to date (pulls latest code; leaves caches and Drive checkpoints untouched).

In [ ]:
import os
REPO_URL = "https://github.com/hamidreza/sudo_rm_rf.git"  # <- change to your repo
REPO_DIR = "/content/sudo_rm_rf"

if not os.path.exists(REPO_DIR):
    os.system('git clone {REPO_URL} {REPO_DIR}'.format(REPO_URL=REPO_URL, REPO_DIR=REPO_DIR))
else:
    os.system('cd {DIR} && git pull --ff-only'.format(DIR=REPO_DIR))

os.chdir(os.path.join(REPO_DIR, 'sudo_rm_rf/dnn/experiments'))  # runners expect this CWD
print('CWD:', os.getcwd())

## 4. Mount Google Drive

Only checkpoints / logs / metrics live on the Drive.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/farsi_wham_experiments'
DRIVE_CHECKPOINTS = os.path.join(DRIVE_ROOT, 'checkpoints')
DRIVE_LOGS = os.path.join(DRIVE_ROOT, 'experiment_logs')
DRIVE_METRICS = os.path.join(DRIVE_ROOT, 'metrics')
for d in (DRIVE_CHECKPOINTS, DRIVE_LOGS, DRIVE_METRICS):
    os.makedirs(d, exist_ok=True)

LATEST_CKPT = os.path.join(DRIVE_CHECKPOINTS, 'latest_checkpoint.pt')
print('Drive experiment root:', DRIVE_ROOT)
print('Resumable:', os.path.exists(LATEST_CKPT), '->', LATEST_CKPT)

## 5. Rebuild audio caches (every session, ~10-15 min)

Downloads YodaLingua-Farsi + the WHAM noise subset, resamples to 16 kHz, and writes the loader's index JSONs under `/data`. The `datasets` line downloads to the HF cache at `/root/.cache/huggingface`, which is also ephemeral — that is expected, we never reuse it.

In [ ]:
import os
os.system('mkdir -p /data')
os.system('python /content/sudo_rm_rf/sudo_rm_rf/utils/prepare_farsi_wham_cache.py '
          '--speech_out /data/farsi_wham --noise_out /data/wham_noise_16k '
          '--n_jobs 2')

## 6. Train (fresh or resume)

If `latest_checkpoint.pt` exists on the Drive the run resumes from it; otherwise training starts from scratch. Everything is queued to run within this session's lifetime; stop the cell any time — the next session resumes automatically. Note: `latest_checkpoint.pt` is saved after **each epoch**, and dated backups every 2 epochs (`--save_checkpoint_every 2`).

In [ ]:
resume = '-rfc latest' if os.path.exists(LATEST_CKPT) else ''

cmd = f'''python run_farsi_wham_separation.py \\
    --train FARSI_WHAM --val FARSI_WHAM --test FARSI_WHAM \\
    --separation_task sep_noisy \\
    --min_num_sources 1 --max_num_sources 3 --n_channels 1 \\
    --model_type groupcomm_v2 \\
    --enc_kernel_size 41 --enc_num_basis 512 --out_channels 256 \\
    --in_channels 512 --num_blocks 8 --group_size 16 --upsampling_depth 5 \\
    --audio_timelength 4.0 -fs 16000 -bs 8 -lr 0.002 --clip_grad_norm 5.0 \\
    --patience 20 \\
    -tags farsi_wham_gc \\
    --n_epochs 100 --n_train 20000 \\
    --checkpoints_path {DRIVE_CHECKPOINTS} --save_checkpoint_every 2 \\
    --experiment_logs_path {DRIVE_LOGS} \\
    --metrics_logs_path {DRIVE_METRICS} \\
    --n_jobs 2 \\
    {resume} \\
'''
print('' if resume else 'Fresh run.\\n')
os.system(cmd)

## 7. Inspect metrics (no comet_ml needed)

Each epoch appends a JSON line with the training loss and the ABSOLUTE SI-SDR of every `1/2/3-speaker` validation bucket to `metrics.jsonl` on the Drive.

In [ ]:
import json

metrics_path = os.path.join(DRIVE_METRICS, 'metrics.jsonl')
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        rows = [json.loads(line) for line in f]
    print('epochs so far:', len(rows))
    for row in rows[-5:]:
        losses = {k: round(v['mean'], 2) for k, v in row['losses'].items()}
        print('epoch {:>3} lr {:>9} '.format(row['epoch'], round(row['lr'], 8)), losses)
else:
    print('No metrics yet — run the training cell first.')

## Notes

- **resume correctness** throughout: the runner writes `latest_checkpoint.pt` to `--checkpoints_path` right after each epoch, so as long as Colab sessions end after an epoch boundary (e.g. stop overnight at the end of an epoch), you lose nothing but the in-progress epoch.
- If you use a free Colab disk without the room to hold ~24 GB of wav files, force the noise subset off with `--n_train` bounded lower, or pass `--speech_only`/`--noise_only` to Stage 5 selectively.
- The runner's audio samples (10 val examples per bucket) are written under `{DRIVE_LOGS}` each epoch.
- `--val`/`--test` FARSI_WHAM flags are documentary: the runner always builds val/test buckets from the FARSI_WHAM loader with `n_estimated_sources=3` (PIT over all estimates).
- If Colab drops to a CPU-only runtime, the runner will fail at `model.cuda()` — request a GPU runtime before rerunning; with `-rfc latest` you resume from the last epoch.